# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API Key exists and begins sk-proj-


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [5]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [6]:
get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

In [7]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [8]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [9]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [10]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [11]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [13]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7882
* To create a public link, set `share=True` in `launch()`.


Tool called for city washington
Tool called for city paris
Tool called for city London


## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [14]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [15]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [16]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7883
* To create a public link, set `share=True` in `launch()`.


Tool called for city Paris
Tool called for city tokyo
Tool called for city london
Tool called for city berlin
Tool called for city London
Tool called for city Paris


In [17]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [19]:
import sqlite3


In [20]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS prices (
            city TEXT PRIMARY KEY,        -- display name
            city_search TEXT NOT NULL,    -- normalized version
            price REAL NOT NULL
        )
    """)
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_city_search ON prices(city_search)")
    conn.commit()


In [21]:
import re

def normalize(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)  # remove punctuation
    text = text.replace(" ", "")        # remove spaces
    return text

In [32]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)

    city_search = normalize(city)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        cursor.execute("""
            SELECT city, price
            FROM prices
            WHERE city_search LIKE ?
        """, (f"%{city_search}%",))

        result = cursor.fetchone()

        if result:
            return f"Ticket price to {result[0].capitalize()} is ${result[1]}"
        else:
            return "No price data available for this city"


In [24]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

In [36]:
def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: Setting price for {city}", flush=True)
    city_display = city.strip()
    city_search = normalize(city)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO prices (city, city_search, price)
            VALUES (?, ?, ?)
            ON CONFLICT(city) DO UPDATE SET
                city_search = excluded.city_search,
                price = excluded.price
        """, (city_display, city_search, price))
        conn.commit()


In [28]:
import json

# Open and read the JSON file
with open("ticket_prices.json", "r") as file:
    ticket_prices = json.load(file)

# Example usage
print(ticket_prices["London".lower()])
print(ticket_prices["Tokyo".lower()])
print(ticket_prices)


799
1420
{'abu dhabi': 1100, 'abuja': 1150, 'accra': 1100, 'addis ababa': 1200, 'algiers': 980, 'amman': 1000, 'amsterdam': 890, 'ankara': 1020, 'antananarivo': 1450, 'apia': 2400, 'ashgabat': 1400, 'asmara': 1250, 'astana': 1200, 'asunción': 1500, 'athens': 920, 'baghdad': 1200, 'baku': 1050, 'bamako': 1200, 'bandar seri begawan': 1400, 'bangkok': 1150, 'bangui': 1350, 'banjul': 1200, 'basseterre': 1200, 'beijing': 1350, 'beirut': 1020, 'belgrade': 920, 'belmopan': 1200, 'berlin': 870, 'bern': 920, 'bishkek': 1250, 'bogotá': 1450, 'brasilia': 1500, 'bratislava': 900, 'brazzaville': 1360, 'bridgetown': 1250, 'brussels': 880, 'bucharest': 910, 'budapest': 890, 'buenos aires': 1600, 'cairo': 980, 'canberra': 2800, 'caracas': 1500, 'castries': 1200, 'chișinău': 980, 'conakry': 1250, 'copenhagen': 930, 'dakar': 1200, 'damascus': 1300, 'dhaka': 1020, 'djibouti': 1250, 'dodoma': 1200, 'doha': 1100, 'dublin': 850, 'dushanbe': 1300, 'freetown': 1250, 'funafuti': 2600, 'gaborone': 1350, 'george

In [29]:
# ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [33]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [34]:
get_ticket_price("Washington")

DATABASE TOOL CALLED: Getting price for Washington


'Ticket price to Washington, d.c. is $650.0'

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7884
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for London
DATABASE TOOL CALLED: Getting price for Paris
DATABASE TOOL CALLED: Getting price for Berlin
DATABASE TOOL CALLED: Getting price for Tokyo
DATABASE TOOL CALLED: Getting price for Abu dhabi
DATABASE TOOL CALLED: Getting price for Amsterdam
DATABASE TOOL CALLED: Getting price for Newyork city


## Exercise

Add a tool to set the price of a ticket!

In [37]:
# Tool definition for setting price
set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to a destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city to set the ticket price for",
            },
            "price": {
                "type": "number",
                "description": "The price of a return ticket to the city",
            },
        },
        "required": ["city", "price"],
        "additionalProperties": False,
    },
}

# Registry mapping tool names to their handler functions
TOOL_REGISTRY = {
    "get_ticket_price": lambda args: get_ticket_price(args["destination_city"]),
    "set_ticket_price": lambda args: set_ticket_price(args["city"], args["price"]),
}

# All tool schemas in one list for passing to the API
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function},
]


def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        handler = TOOL_REGISTRY.get(name)
        if handler:
            result = handler(arguments)
        else:
            result = f"Unknown tool: {name}"

        responses.append({
            "role": "tool",
            "content": str(result) if result else "Done",
            "tool_call_id": tool_call.id,
        })
    return responses

In [38]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7885
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for London
DATABASE TOOL CALLED: Getting price for Paris
DATABASE TOOL CALLED: Getting price for Berlin
DATABASE TOOL CALLED: Getting price for Tokyo
DATABASE TOOL CALLED: Getting price for Dubai
DATABASE TOOL CALLED: Getting price for Newyork city
DATABASE TOOL CALLED: Getting price for Dallas Texas
DATABASE TOOL CALLED: Setting price for Dubai
DATABASE TOOL CALLED: Setting price for New York City
DATABASE TOOL CALLED: Setting price for Dallas Texas
DATABASE TOOL CALLED: Getting price for New York City
DATABASE TOOL CALLED: Getting price for Dallas Texas
DATABASE TOOL CALLED: Getting price for Washington
DATABASE TOOL CALLED: Getting price for Paris
DATABASE TOOL CALLED: Getting price for Newyork
DATABASE TOOL CALLED: Getting price for Dallas Texas


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>